In [ ]:
# FPGA-Accelerated Inference for PYNQ

import numpy as np
import json
from pynq import Overlay, allocate

# Load quantized artifacts (pure NumPy with no torch required)
w_fc1 = np.load("artifacts/fc1_weight.npy") # shape (128, 784), int8
b_fc1 = np.load("artifacts/fc1_bias.npy") # shape (128,), float32
with open("artifacts/weight_scales.json") as f:
    w_scale_fc1 = json.load(f)["fc1"]
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)

# Load overlay
FC1_M = 1
FC1_K = 784
FC1_N = 128

ol = Overlay("block_design.bit") # corret .bit name

ip = ol.multiply_0 # corrent IP core instance name

# Allocate DMA-capable buffers
A_buf = allocate(shape=(FC1_M * FC1_K,), dtype=np.int8)
B_buf = allocate(shape=(FC1_K * FC1_N,), dtype=np.int8)
C_buf = allocate(shape=(FC1_M * FC1_N,), dtype=np.int32)

# Load one MNIST sample
images = np.load("artifacts/mnist_test_images.npy") # shape (10000, 28, 28), float32 [0,1]
labels = np.load("artifacts/mnist_test_labels.npy") # shape (10000,)
label = int(labels[0])

x_fp32 = images[0].reshape(-1) # shape (784,)
x_int8 = np.clip(np.round(x_fp32 / act_scales["input"]), -128, 127).astype(np.int8)

# Fill buffers
A_buf[:] = x_int8
B_buf[:] = w_fc1.T.flatten() # transpose to (K, N) = (784, 128) row-major

A_buf.sync_to_device()
B_buf.sync_to_device()

# Configure IP registers
ip.write(0x10, A_buf.device_address & 0xFFFFFFFF)
ip.write(0x14, (A_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x1c, B_buf.device_address & 0xFFFFFFFF)
ip.write(0x20, (B_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x28, C_buf.device_address & 0xFFFFFFFF)
ip.write(0x2c, (C_buf.device_address >> 32) & 0xFFFFFFFF)
ip.write(0x34, FC1_M)
ip.write(0x3c, FC1_K)
ip.write(0x44, FC1_N)

# Start and poll
ip.write(0x00, 0x01)

while (ip.read(0x00) & 0x2) == 0:
    pass

# Retrieve and dequantize
C_buf.sync_from_device()

y_int32 = np.array(C_buf).reshape(FC1_M, FC1_N) # (1, 128) int32
y_fp32  = y_int32.astype(np.float32) * (act_scales["input"] * w_scale_fc1)
y_fp32 += b_fc1 # add bias in float
y_relu  = np.maximum(y_fp32, 0.0) # ReLU then feed to FC2

print(f"Label: {label}, FC1 output (first 8): {y_relu[0, :8]}")

A_buf.close()
B_buf.close()
C_buf.close()

In [ ]:
# Compare FPGA and CPU Results

import numpy as np
import json

# Reload artifacts
params = {
    "fc1": {
        "weight_int8": np.load("artifacts/fc1_weight.npy"),
        "bias_fp32":   np.load("artifacts/fc1_bias.npy"),
    },
}
with open("artifacts/activation_scales.json") as f:
    act_scales = json.load(f)
with open("artifacts/weight_scales.json") as f:
    weight_scales = json.load(f)

images = np.load("artifacts/mnist_test_images.npy")
x = images[0].reshape(1, -1)

# Software FC1
x_int8 = np.clip(np.round(x / act_scales["input"]), -128, 127).astype(np.int8)
y_int32 = x_int8.astype(np.int32) @ params["fc1"]["weight_int8"].T.astype(np.int32)
y_fp32  = y_int32.astype(np.float32) * (act_scales["input"] * weight_scales["fc1"])
y_fp32 += params["fc1"]["bias_fp32"]
y_relu_sw = np.maximum(y_fp32, 0.0)

print("Software FC1 (first 8):", y_relu_sw[0, :8])
print("FPGA    FC1 (first 8):", y_relu[0, :8])
print("Max diff:", np.max(np.abs(y_relu_sw[0] - y_relu[0])))